## Cell 0: RUNTIME CONFIGURATION AND OVERVIEW


In [ ]:
ADAPTER_PATH="/ocean/projects/cis260137p/ssingh40/qwen3_run/outcome_aware"

In [2]:
# import os
# os.environ["HF_HOME"]="/ocean/projects/cis260137p/ssingh40/hf_cache"
# os.environ["HF_HUB_CACHE"]="/ocean/projects/cis260137p/ssingh40/hf_cache/hub"
# os.environ["TRANSFORMERS_CACHE"]="/ocean/projects/cis260137p/ssingh40/hf_cache/transformers"
# os.environ["HF_HUB_DISABLE_XET"]="1"

In [3]:
import os, torch
from transformers import Qwen3VLForConditionalGeneration,AutoProcessor
from peft import PeftModel

CACHE_DIR="/ocean/projects/cis260137p/ssingh40/hf_cache"
os.environ["HF_HOME"]=CACHE_DIR
os.environ["HF_HUB_CACHE"]=CACHE_DIR+"/hub"
os.environ["TRANSFORMERS_CACHE"]=CACHE_DIR+"/transformers"
os.environ["HF_HUB_DISABLE_XET"]="1"

BASE_MODEL_ID="Qwen/Qwen3-VL-8B-Thinking"
ADAPTER_PATH="/ocean/projects/cis260137p/ssingh40/qwen3_run/outcome_aware"

base_model=Qwen3VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="sdpa",
    cache_dir=CACHE_DIR,
)

model=PeftModel.from_pretrained(base_model,ADAPTER_PATH)
model=model.merge_and_unload()

processor=AutoProcessor.from_pretrained(
    BASE_MODEL_ID,
    cache_dir=CACHE_DIR,
    min_pixels=256*28*28,
    max_pixels=768*28*28,
)

model.eval()

/ocean/projects/cis260137p/ssingh40/shared_envs/vlm_oa/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████| 750/750 [01:07<00:00, 11.11it/s]


Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1152)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1152, out_features=3456, bias=True)
            (proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (linear_fc2): Linear(in_features=4304, out_features=1152, bias=True)
            (act_fn): GELUTanh()
          )
        )
      )
 

In [4]:
print(type(model))
print(model)

<class 'transformers.models.qwen3_vl.modeling_qwen3_vl.Qwen3VLForConditionalGeneration'>
Qwen3VLForConditionalGeneration(
  (model): Qwen3VLModel(
    (visual): Qwen3VLVisionModel(
      (patch_embed): Qwen3VLVisionPatchEmbed(
        (proj): Conv3d(3, 1152, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 1152)
      (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-26): 27 x Qwen3VLVisionBlock(
          (norm1): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((1152,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3VLVisionAttention(
            (qkv): Linear(in_features=1152, out_features=3456, bias=True)
            (proj): Linear(in_features=1152, out_features=1152, bias=True)
          )
          (mlp): Qwen3VLVisionMLP(
            (linear_fc1): Linear(in_features=1152, out_features=4304, bias=True)
            (linear_fc2): Linear(in_features=4304, out_f

In [5]:
# Colab Runtime: GPU → A100 (preferred) or L4
# Model: Qwen/Qwen3-VL-8B-Thinking via HuggingFace Transformers
# Dataset: osunlp/Multimodal-Mind2Web (test_website split)
# Ablation 1: HTML + Image (zero-shot)
# Ablation 2: HTML + Image + Few-shot (3 balanced examples)
#
# Backend: HuggingFace Transformers with Qwen3VLForConditionalGeneration
#   - bf16 precision
#   - flash_attention_2 if available (falls back to sdpa)
#   - device_map="auto"
#
# Top-3 strategy: Per-candidate scoring via conditional log-likelihood.
#   For each candidate action, we compute the log-probability of the
#   model generating "Answer: <index>" given the prompt. We rank all
#   candidates by score to get top-1 and top-3 predictions. This is
#   more reliable than parsing free-form generation for top-3.
#
# Expected VRAM: ~18-22GB for bf16 8B model + image processing.
# Expected runtime: ~2-4 hours for 1019 examples on A100 (per ablation).
# ============================================================


In [6]:
# import json
# import re
# import os
# from collections import defaultdict
# from transformers import AutoTokenizer

# PREDICTIONS_PATH = "/content/zero_shot_predictions.json"
# OUTPUT_PREDICTIONS_PATH = "/content/zero_shot_predictions_normalized.json"
# OUTPUT_METRICS_PATH = "/content/zero_shot_metrics.json"
# MODEL_ID = "Qwen/Qwen3-VL-8B-Thinking"

# tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# with open(PREDICTIONS_PATH, "r") as f:
#     predictions = json.load(f)

# print(f"Loaded {len(predictions)} predictions")

# token_length_cache = {}
# def get_answer_token_length(index):
#     if index not in token_length_cache:
#         text = f"Answer: {index}"
#         ids = tokenizer.encode(text, add_special_tokens=False)
#         token_length_cache[index] = len(ids)
#     return token_length_cache[index]

# fixed_count = 0
# changed_count = 0

# for p in predictions:
#     raw = p.get("raw_model_output", "")
#     if not raw.startswith("{"):
#         continue

#     try:
#         data = json.loads(raw)
#     except json.JSONDecodeError:
#         continue

#     scores = data.get("scores", {})
#     if not scores:
#         continue

#     normalized_scores = {}
#     for idx_str, score in scores.items():
#         idx = int(idx_str)
#         tok_len = get_answer_token_length(idx)
#         normalized_scores[idx] = score / tok_len

#     ranked = sorted(normalized_scores.keys(), key=lambda i: normalized_scores[i], reverse=True)

#     old_pred = p["predicted_index"]
#     new_pred = ranked[0]
#     new_top3 = ranked[:3]

#     candidates = p["candidate_actions"]
#     new_pred_action = candidates[new_pred] if 0 <= new_pred < len(candidates) else "INVALID"
#     new_top3_actions = [candidates[j] if 0 <= j < len(candidates) else "INVALID" for j in new_top3]

#     if old_pred != new_pred:
#         changed_count += 1

#     p["predicted_index"] = new_pred
#     p["predicted_action"] = new_pred_action
#     p["top3_predicted_indices"] = new_top3
#     p["top3_predicted_actions"] = new_top3_actions
#     p["raw_model_output_normalized"] = json.dumps({
#         "normalized_scores": {str(k): round(v, 4) for k, v in normalized_scores.items()}
#     })

#     fixed_count += 1

# print(f"Re-ranked {fixed_count} predictions")
# print(f"Changed top-1 prediction for {changed_count} examples")

# def extract_op_and_value_from_repr(action_repr):
#     m = re.search(r"->\s*(CLICK|TYPE|SELECT)\s*(.*)", action_repr)
#     if m:
#         op_type = m.group(1).strip()
#         value = m.group(2).strip()
#         return op_type, value
#     return action_repr.strip(), ""

# def char_f1(pred_str, gold_str):
#     pred_chars = list(pred_str)
#     gold_chars = list(gold_str)
#     if len(pred_chars) == 0 and len(gold_chars) == 0:
#         return 1.0
#     if len(pred_chars) == 0 or len(gold_chars) == 0:
#         return 0.0
#     common = 0
#     gold_remaining = list(gold_chars)
#     for c in pred_chars:
#         if c in gold_remaining:
#             common += 1
#             gold_remaining.remove(c)
#     precision = common / len(pred_chars) if pred_chars else 0
#     recall = common / len(gold_chars) if gold_chars else 0
#     if precision + recall == 0:
#         return 0.0
#     return 2 * precision * recall / (precision + recall)

# def compute_operation_f1(pred_action_repr, gold_action_repr):
#     pred_op, pred_val = extract_op_and_value_from_repr(pred_action_repr)
#     gold_op, gold_val = extract_op_and_value_from_repr(gold_action_repr)
#     pred_combined = f"{pred_op} {pred_val}".strip()
#     gold_combined = f"{gold_op} {gold_val}".strip()
#     return char_f1(pred_combined, gold_combined)

# n = len(predictions)
# ele_acc_sum = 0
# op_f1_sum = 0
# step_sr_sum = 0
# top3_sum = 0
# task_steps = defaultdict(list)

# for p in predictions:
#     gold_idx = p["gold_target_index"]
#     pred_idx = p["predicted_index"]
#     candidates = p["candidate_actions"]
#     gold_repr = p["gold_target_action"]
#     pred_repr = candidates[pred_idx] if 0 <= pred_idx < len(candidates) else ""

#     ea = int(pred_idx == gold_idx)
#     of1 = compute_operation_f1(pred_repr, gold_repr)
#     ssr = int(pred_idx == gold_idx and of1 >= 0.9)
#     t3 = int(gold_idx in p.get("top3_predicted_indices", [pred_idx]))

#     ele_acc_sum += ea
#     op_f1_sum += of1
#     step_sr_sum += ssr
#     top3_sum += t3

#     p["metric_ele_acc"] = ea
#     p["metric_op_f1"] = of1
#     p["metric_step_sr"] = ssr
#     p["metric_top3_acc"] = t3

#     task_id = p.get("task_id", "unknown")
#     task_steps[task_id].append(ssr)

# task_sr_sum = 0
# for task_id, steps in task_steps.items():
#     task_sr_sum += int(all(s == 1 for s in steps))

# num_tasks = len(task_steps)

# metrics = {
#     "element_accuracy": ele_acc_sum / n,
#     "operation_f1": op_f1_sum / n,
#     "step_success_rate": step_sr_sum / n,
#     "task_success_rate": task_sr_sum / num_tasks if num_tasks > 0 else 0,
#     "top3_accuracy": top3_sum / n,
#     "num_examples": n,
#     "num_tasks": num_tasks,
#     "scoring": "length_normalized",
# }

# print("\n=== Zero-Shot Metrics (Length-Normalized) ===")
# for k, v in metrics.items():
#     print(f"  {k}: {v}")

# with open(OUTPUT_PREDICTIONS_PATH, "w") as f:
#     json.dump(predictions, f, indent=2, default=str)
# print(f"\nSaved: {OUTPUT_PREDICTIONS_PATH}")

# with open(OUTPUT_METRICS_PATH, "w") as f:
#     json.dump(metrics, f, indent=2)
# print(f"Saved: {OUTPUT_METRICS_PATH}")

In [7]:
# !zip /content/zero_shot_results.zip /content/zero_shot_predictions_normalized.json /content/zero_shot_metrics.json
# from google.colab import files
# files.download('/content/zero_shot_results.zip')

In [ ]:
import os, getpass
os.environ["HF_TOKEN"] = getpass.getpass("Enter you HF token:")

## Cell 1: INSTALL PACKAGES


In [ ]:
# !pip install -q "transformers>=4.57.0" accelerate datasets pillow torch qwen-vl-utils


## Cell 2: IMPORTS


In [9]:
import os
import json
import re
import time
import math
import gc
import traceback
from io import BytesIO
from collections import defaultdict
from datetime import datetime

import torch
import numpy as np
from PIL import Image
from datasets import load_dataset


## Cell 3: CONFIGURATION


In [10]:
CONFIG = {
    "model_id":"Qwen/Qwen3-VL-8B-Thinking",
    "dataset_id":"osunlp/Multimodal-Mind2Web",
    "eval_split":"test_website",
    "train_split":"train",
    "max_html_chars":4000,
    "max_new_tokens":512,
    "sanity_check_n":10,
    "output_dir":"/ocean/projects/cis260137p/ssingh40/qwen3_run/ablation_outputs/outcomeaware",
    "use_bf16":True,
    "use_flash_attn":False,
    "use_candidate_scoring":True,
    "scoring_max_new_tokens":10,
    "timestamp":datetime.now().strftime("%Y%m%d_%H%M%S"),
}

os.makedirs(CONFIG["output_dir"],exist_ok=True)

with open(os.path.join(CONFIG["output_dir"],"run_config.json"), "w") as f:
    json.dump(CONFIG,f,indent=2)


## Cell 4: LOAD DATASET


In [11]:

ds_test = load_dataset(CONFIG["dataset_id"],split=CONFIG["eval_split"])
ds_train = load_dataset(CONFIG["dataset_id"],split=CONFIG["train_split"])


## Cell 5: INSPECT DATASET SCHEMA


In [12]:
print("Dataset Features")
print(ds_test.features)
print()

ex=ds_test[0]

print("Example Keys")
for k,v in ex.items():
    if isinstance(v,str):
        print(f"{k}: str,len={len(v)},preview={v[:120]}...")
    elif isinstance(v,list):
        first=str(v[0])[:120] if len(v)>0 else "EMPTY"
        print(f"{k}: list,len={len(v)},first={first}...")
    elif isinstance(v,Image.Image):
        print(f"{k}: PIL.Image,size={v.size},mode={v.mode}")
    elif isinstance(v,dict):
        print(f"{k}: dict,keys={list(v.keys())}")
    else:
        print(f"{k}: {type(v).__name__},value={str(v)[:120]}")

print()
print("operation field")
op=json.loads(ex["operation"]) if isinstance(ex["operation"],str) else ex["operation"]
print(json.dumps(op,indent=2))

print()
print("action_reprs first 5")
for i,at in enumerate(ex["action_reprs"][:5]):
    print(f"[{i}] {at}")

print()
print("target_action_index")
print(ex["target_action_index"])

print()
print("target_action_reprs")
print(ex["target_action_reprs"])

Dataset Features
{'action_uid': Value('string'), 'raw_html': Value('string'), 'cleaned_html': Value('string'), 'operation': Value('string'), 'pos_candidates': List(Value('string')), 'neg_candidates': List(Value('string')), 'website': Value('string'), 'domain': Value('string'), 'subdomain': Value('string'), 'annotation_id': Value('string'), 'confirmed_task': Value('string'), 'screenshot': Image(mode=None, decode=True), 'action_reprs': List(Value('string')), 'target_action_index': Value('string'), 'target_action_reprs': Value('string')}



Example Keys
action_uid: str,len=36,preview=79c4a963-4aa9-49c1-9257-6b0d5069c551...
raw_html: str,len=85102,preview=<html backend_node_id="113">
  <body backend_node_id="188">
    <div backend_node_id="189">
      <div backend_node_id="...
cleaned_html: str,len=85102,preview=<html backend_node_id="113">
  <body backend_node_id="188">
    <div backend_node_id="189">
      <div backend_node_id="...
operation: str,len=52,preview={"original_op": "CLICK", "value": "", "op": "CLICK"}...
pos_candidates: list,len=2,first={"tag": "label", "attributes": "{\"backend_node_id\": \"110\", \"bounding_box_rect\": \"356,461,320,34\", \"class\": \"b...
neg_candidates: list,len=491,first={"tag": "div", "attributes": "{\"backend_node_id\": \"189\", \"bounding_box_rect\": \"0,0,1280,1080\", \"id\": \"__next\...
website: str,len=12,preview=tiktok.music...
domain: str,len=13,preview=Entertainment...
subdomain: str,len=5,preview=Music...
annotation_id: str,len=36,preview=013781df-4391-4533-bcb1-15f6819064f6..

## Cell 6: PARSE DATASET HELPERS


In [13]:
def parse_operation(op_field):
    if isinstance(op_field, str):
        return json.loads(op_field)
    return op_field

def get_op_type(op_dict):
    return op_dict.get("op", "CLICK").upper()

def get_op_value(op_dict):
    return op_dict.get("value", "")

def truncate_html(html_str, max_chars):
    if len(html_str) <= max_chars:
        return html_str, False
    half = max_chars // 2
    return html_str[:half] + "\n... [TRUNCATED] ...\n" + html_str[-half:], True

def get_candidate_actions(example):
    return example["action_reprs"]

def get_target_index(example):
    idx_str = example["target_action_index"]
    try:
        return int(idx_str)
    except (ValueError, TypeError):
        return -1

def get_target_repr(example):
    return example["target_action_reprs"]

def get_screenshot(example):
    img = example["screenshot"]
    if isinstance(img, Image.Image):
        return img
    if isinstance(img, dict) and "bytes" in img:
        return Image.open(BytesIO(img["bytes"]))
    if isinstance(img, dict) and "path" in img:
        return Image.open(img["path"])
    return None

def get_task_id(example):
    return example["annotation_id"]


ex= ds_test[0]
op=parse_operation(ex["operation"])
img=get_screenshot(ex)


## Cell 7: SELECT FEW-SHOT EXAMPLES


## Cell 8: PROMPT CONSTRUCTION


In [14]:
SYSTEM_PROMPT=(
    "You are a web navigation agent. You are given a screenshot of a webpage, "
    "the cleaned HTML of the page, and a list of candidate actions. "
    "Your job is to select the single best action that accomplishes the user's task. "
    "Do NOT invent new actions. You MUST choose from the provided candidates only.\n\n"
    "Respond with exactly:\nAnswer: <index>\n"
    "where <index> is the zero-based integer index of your chosen action."
)

def truncate_html(html_str,max_chars):
    if len(html_str)<=max_chars:
        return html_str,False
    half=max_chars//2
    return html_str[:half]+"\n... [TRUNCATED] ...\n"+html_str[-half:],True

def build_candidate_str(candidates):
    return "\n".join([f"[{i}] {c}" for i,c in enumerate(candidates)])

def build_zeroshot_prompt(example,max_html_chars):
    html_raw=example["cleaned_html"]
    html_text,was_truncated=truncate_html(html_raw,max_html_chars)
    cand_str=build_candidate_str(get_candidate_actions(example))
    task=example["confirmed_task"]
    trunc_note=f" (truncated from {len(html_raw)} to {max_html_chars} chars)" if was_truncated else ""

    text=(
        f"Task: {task}\n\n"
        f"Cleaned HTML{trunc_note}:\n{html_text}\n\n"
        f"Candidate Actions:\n{cand_str}\n\n"
        f"Select the correct action index. Respond with exactly:\nAnswer: <index>"
    )
    return text,html_text

def build_messages_zeroshot(example,max_html_chars):
    text_prompt,used_html=build_zeroshot_prompt(example,max_html_chars)
    img=get_screenshot(example)
    content=[]
    if img is not None:
        content.append({"type":"image","image":img.convert("RGB")})
    content.append({"type":"text","text":text_prompt})
    messages=[
        {"role":"system","content":[{"type":"text","text":SYSTEM_PROMPT}]},
        {"role":"user","content":content},
    ]
    return messages,used_html

## Cell 9: LOAD MODEL


In [15]:
# print("Loading model...")

# from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

# attn_impl = "flash_attention_2" if CONFIG["use_flash_attn"] else "sdpa"

# try:
#     model = Qwen3VLForConditionalGeneration.from_pretrained(
#         CONFIG["model_id"],
#         torch_dtype=torch.bfloat16 if CONFIG["use_bf16"] else torch.float16,
#         attn_implementation=attn_impl,
#         device_map="auto",
#         low_cpu_mem_usage=True,
#     )
#     print(f"Model loaded with attn_implementation={attn_impl}")
# except Exception as e:
#     print(f"Failed with {attn_impl}: {e}")
#     print("Falling back to sdpa...")
#     model = Qwen3VLForConditionalGeneration.from_pretrained(
#         CONFIG["model_id"],
#         torch_dtype=torch.bfloat16 if CONFIG["use_bf16"] else torch.float16,
#         attn_implementation="sdpa",
#         device_map="auto",
#         low_cpu_mem_usage=True,
#     )
#     print("Model loaded with attn_implementation=sdpa")

# processor = AutoProcessor.from_pretrained(CONFIG["model_id"])
# model.eval()
# print(f"Model device: {model.device}")
# print(f"Model dtype: {model.dtype}")

In [16]:
# print("Loading base model + LoRA adapter...")

# import torch
# from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
# from peft import PeftModel

# ADAPTER_PATH = "/content/ocean/projects/cis260137p/abhyanka/outputs/qwen-vl-next-action-sft/checkpoint-6545"
# attn_impl = "flash_attention_2" if CONFIG["use_flash_attn"] else "sdpa"

# try:
#     base_model = Qwen3VLForConditionalGeneration.from_pretrained(
#         CONFIG["model_id"],
#         torch_dtype=torch.bfloat16 if CONFIG["use_bf16"] else torch.float16,
#         attn_implementation=attn_impl,
#         device_map="auto",
#         low_cpu_mem_usage=True,
#     )
#     print(f"Base model loaded with attn_implementation={attn_impl}")
# except Exception as e:
#     print(f"Failed with {attn_impl}: {e}")
#     print("Falling back to sdpa...")
#     base_model = Qwen3VLForConditionalGeneration.from_pretrained(
#         CONFIG["model_id"],
#         torch_dtype=torch.bfloat16 if CONFIG["use_bf16"] else torch.float16,
#         attn_implementation="sdpa",
#         device_map="auto",
#         low_cpu_mem_usage=True,
#     )
#     print("Base model loaded with attn_implementation=sdpa")

# model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
# model = model.merge_and_unload()

# processor = AutoProcessor.from_pretrained(CONFIG["model_id"])
# model.eval()

# print(f"Model device: {model.device}")
# print(f"Model dtype: {model.dtype}")
# print("LoRA adapter loaded and merged successfully")

from transformers import AutoProcessor

processor=AutoProcessor.from_pretrained(
    CONFIG["model_id"],
    cache_dir="/ocean/projects/cis260137p/ssingh40/hf_cache",
    min_pixels=256*28*28,
    max_pixels=768*28*28,
)

## Cell 10: INFERENCE FUNCTIONS


In [17]:
def build_messages_zeroshot(example,max_html_chars):
    text_prompt,used_html=build_zeroshot_prompt(example,max_html_chars)
    img=get_screenshot(example)
    content=[]
    if img is not None:
        content.append({"type":"image","image":img.convert("RGB")})
    content.append({"type":"text","text":text_prompt})
    messages=[
        {"role":"system","content":[{"type":"text","text":SYSTEM_PROMPT}]},
        {"role":"user","content":content},
    ]
    return messages,used_html

def build_messages_fewshot(example,fewshot_prefix,fewshot_images,max_html_chars):
    text_prompt,used_html=build_fewshot_prompt(example,fewshot_prefix,max_html_chars)
    img=get_screenshot(example)
    content=[]
    for fs_img in fewshot_images:
        if fs_img is not None:
            content.append({"type":"image","image":fs_img.convert("RGB")})
    if img is not None:
        content.append({"type":"image","image":img.convert("RGB")})
    content.append({"type":"text","text":text_prompt})
    messages=[
        {"role":"system","content":[{"type":"text","text":SYSTEM_PROMPT}]},
        {"role":"user","content":content},
    ]
    return messages,used_html

def _split_messages_and_images(messages):
    clean=[]
    imgs=[]
    for msg in messages:
        new_content=[]
        for item in msg["content"]:
            if item.get("type")=="image":
                img=item.get("image")
                if img is not None:
                    imgs.append(img.convert("RGB"))
                new_content.append({"type":"image"})
            else:
                new_content.append(item)
        clean.append({"role":msg["role"],"content":new_content})
    return clean,imgs

def _apply_chat_template(messages):
    clean_messages,imgs=_split_messages_and_images(messages)

    text=processor.apply_chat_template(
        clean_messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs=processor(
        text=[text],
        images=imgs,
        padding=True,
        truncation=False,
        return_tensors="pt",
    )

    inputs={k:v.to(model.device) if hasattr(v,"to") else v for k,v in inputs.items()}
    return inputs

def generate_response(messages,max_new_tokens):
    inputs=_apply_chat_template(messages)
    with torch.no_grad():
        generated_ids=model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    input_len=inputs["input_ids"].shape[1]
    output_ids=generated_ids[0][input_len:]
    output_text=processor.tokenizer.decode(output_ids,skip_special_tokens=True)
    del inputs,generated_ids
    torch.cuda.empty_cache()
    return output_text

def prepare_inputs_once(messages):
    return _apply_chat_template(messages)

def score_candidate_from_kv(prompt_last_logits,prompt_kv,candidate_index):
    answer_text=f"Answer: {candidate_index}"
    answer_ids=processor.tokenizer.encode(answer_text,add_special_tokens=False)

    log_probs=[]
    first_lp=torch.log_softmax(prompt_last_logits,dim=-1)
    log_probs.append(first_lp[answer_ids[0]].item())

    if len(answer_ids)>1:
        ans_prefix=torch.tensor([answer_ids[:-1]],device=model.device)
        with torch.no_grad():
            ans_out=model(
                input_ids=ans_prefix,
                past_key_values=prompt_kv,
                use_cache=False,
            )

        for t in range(len(answer_ids)-1):
            step_lp=torch.log_softmax(ans_out.logits[0,t,:],dim=-1)
            log_probs.append(step_lp[answer_ids[t+1]].item())

        del ans_out

    return sum(log_probs)/len(log_probs)

def run_inference_single(messages,num_candidates,use_scoring=True):
    model.config.use_cache=True

    if use_scoring and num_candidates<=50:
        cached_inputs=prepare_inputs_once(messages)

        with torch.no_grad():
            prompt_out=model(**cached_inputs,use_cache=True)

        prompt_last_logits=prompt_out.logits[0,-1,:]
        prompt_kv=prompt_out.past_key_values

        scores=[]
        for c_idx in range(num_candidates):
            s=score_candidate_from_kv(prompt_last_logits,prompt_kv,c_idx)
            scores.append(s)

        ranked=sorted(range(num_candidates),key=lambda i:scores[i],reverse=True)
        top1=ranked[0]
        top3=ranked[:3]
        raw_output=json.dumps({"scores":{str(i):round(scores[i],4) for i in range(num_candidates)}})

        del prompt_out,cached_inputs,prompt_kv,prompt_last_logits
        gc.collect()
        torch.cuda.empty_cache()

        return top1,top3,raw_output,scores

    raw_output=generate_response(messages,CONFIG["max_new_tokens"])
    parsed=parse_answer(raw_output,num_candidates)
    return parsed,[parsed],raw_output,[]

def parse_answer(text,num_candidates):
    patterns=[
        r"Answer\s*:\s*(\d+)",
        r"answer\s*:\s*(\d+)",
        r'"Answer"\s*:\s*"?(\d+)"?',
        r"'Answer'\s*:\s*'?(\d+)'?",
    ]
    for pat in patterns:
        m=re.search(pat,text)
        if m:
            idx=int(m.group(1))
            if 0<=idx<num_candidates:
                return idx

    numbers=re.findall(r"\b(\d+)\b",text)
    for n_str in reversed(numbers):
        idx=int(n_str)
        if 0<=idx<num_candidates:
            return idx
    return -1

print("Inference functions defined.")

Inference functions defined.


## Cell 11: METRIC COMPUTATION


In [18]:
def compute_element_accuracy(pred_idx, gold_idx):
    return int(pred_idx == gold_idx)

def char_f1(pred_str, gold_str):
    pred_chars = list(pred_str)
    gold_chars = list(gold_str)
    if len(pred_chars) == 0 and len(gold_chars) == 0:
        return 1.0
    if len(pred_chars) == 0 or len(gold_chars) == 0:
        return 0.0
    common = 0
    gold_remaining = list(gold_chars)
    for c in pred_chars:
        if c in gold_remaining:
            common += 1
            gold_remaining.remove(c)
    precision = common / len(pred_chars) if pred_chars else 0
    recall = common / len(gold_chars) if gold_chars else 0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)

def extract_op_and_value_from_repr(action_repr):
    m = re.search(r"->\s*(CLICK|TYPE|SELECT)\s*(.*)", action_repr)
    if m:
        op_type = m.group(1).strip()
        value = m.group(2).strip()
        return op_type, value
    return action_repr.strip(), ""

def compute_operation_f1(pred_action_repr, gold_action_repr):
    pred_op, pred_val = extract_op_and_value_from_repr(pred_action_repr)
    gold_op, gold_val = extract_op_and_value_from_repr(gold_action_repr)
    pred_combined = f"{pred_op} {pred_val}".strip()
    gold_combined = f"{gold_op} {gold_val}".strip()
    return char_f1(pred_combined, gold_combined)

def compute_step_sr(pred_idx, gold_idx, pred_action_repr, gold_action_repr):
    if pred_idx != gold_idx:
        return 0
    f1 = compute_operation_f1(pred_action_repr, gold_action_repr)
    return int(f1 >= 0.9)

def compute_top3_accuracy(top3_indices, gold_idx):
    return int(gold_idx in top3_indices)

def compute_metrics(predictions):
    n = len(predictions)
    if n == 0:
        return {}
    ele_acc_sum = 0
    op_f1_sum = 0
    step_sr_sum = 0
    top3_sum = 0
    task_steps = defaultdict(list)
    for p in predictions:
        gold_idx = p["gold_target_index"]
        pred_idx = p["predicted_index"]
        candidates = p["candidate_actions"]
        gold_repr = p["gold_target_action"]
        pred_repr = candidates[pred_idx] if 0 <= pred_idx < len(candidates) else ""
        ea = compute_element_accuracy(pred_idx, gold_idx)
        of1 = compute_operation_f1(pred_repr, gold_repr)
        ssr = compute_step_sr(pred_idx, gold_idx, pred_repr, gold_repr)
        t3 = compute_top3_accuracy(p.get("top3_predicted_indices", [pred_idx]), gold_idx)
        ele_acc_sum += ea
        op_f1_sum += of1
        step_sr_sum += ssr
        top3_sum += t3
        p["metric_ele_acc"] = ea
        p["metric_op_f1"] = of1
        p["metric_step_sr"] = ssr
        p["metric_top3_acc"] = t3
        task_id = p.get("task_id", "unknown")
        task_steps[task_id].append(ssr)
    task_sr_sum = 0
    for task_id, steps in task_steps.items():
        task_sr_sum += int(all(s == 1 for s in steps))
    num_tasks = len(task_steps)
    metrics = {
        "element_accuracy": ele_acc_sum / n,
        "operation_f1": op_f1_sum / n,
        "step_success_rate": step_sr_sum / n,
        "task_success_rate": task_sr_sum / num_tasks if num_tasks > 0 else 0,
        "top3_accuracy": top3_sum / n,
        "num_examples": n,
        "num_tasks": num_tasks,
    }
    return metrics

print("Metric functions defined.")

Metric functions defined.


## Cell 12: MAIN INFERENCE LOOP


In [19]:
def run_ablation(dataset,ablation_name,mode="zero_shot",n_examples=None,fewshot_prefix_str=None,fewshot_images=None):
    predictions=[]
    total=len(dataset) if n_examples is None else min(n_examples,len(dataset))
    use_scoring=CONFIG["use_candidate_scoring"]
    fewshot_images=[] if fewshot_images is None else fewshot_images

    print(f"\n{'='*60}")
    print(f"Running ablation: {ablation_name} ({mode})")
    print(f"Examples: {total}, Scoring: {use_scoring}")
    print(f"{'='*60}\n")

    start_time=time.time()

    for i in range(total):
        ex=dataset[i]
        candidates=get_candidate_actions(ex)
        gold_idx=get_target_index(ex)
        gold_repr=get_target_repr(ex)
        task_id=get_task_id(ex)
        html_raw=ex["cleaned_html"]

        if mode=="zero_shot":
            messages,used_html=build_messages_zeroshot(ex,CONFIG["max_html_chars"])
        else:
            messages,used_html=build_messages_fewshot(
                ex,fewshot_prefix_str,fewshot_images,CONFIG["max_html_chars"]
            )

        try:
            pred_idx,top3,raw_output,scores=run_inference_single(
                messages,len(candidates),use_scoring=use_scoring
            )
        except torch.OutOfMemoryError as e:
            print(f"  OOM on example {i}: {e}")
            traceback.print_exc()
            gc.collect()
            torch.cuda.empty_cache()
            raise e
        except Exception as e:
            print(f"  ERROR on example {i}: {e}")
            traceback.print_exc()
            pred_idx=-1
            top3=[-1]
            raw_output=f"ERROR: {str(e)}"
            scores=[]

        top3_actions=[candidates[j] if 0<=j<len(candidates) else "INVALID" for j in top3]
        pred_action=candidates[pred_idx] if 0<=pred_idx<len(candidates) else "INVALID"

        record={
            "example_index":i,
            "action_uid":ex["action_uid"],
            "task_id":task_id,
            "website":ex.get("website",""),
            "confirmed_task":ex["confirmed_task"],
            "truncated_html_len":len(used_html),
            "original_html_len":len(html_raw),
            "was_truncated":len(used_html)<len(html_raw),
            "num_candidates":len(candidates),
            "candidate_actions":candidates,
            "gold_target_index":gold_idx,
            "gold_target_action":gold_repr,
            "predicted_index":pred_idx,
            "predicted_action":pred_action,
            "top3_predicted_indices":top3,
            "top3_predicted_actions":top3_actions,
            "raw_model_output":raw_output[:2000],
            "scores":scores,
        }
        predictions.append(record)

        elapsed=time.time()-start_time
        avg_time=elapsed/(i+1)
        eta=avg_time*(total-i-1)
        correct_str="✓" if pred_idx==gold_idx else "✗"
        print(f"  [{i+1}/{total}] {correct_str} pred={pred_idx} gold={gold_idx} "
              f"| {avg_time:.1f}s/ex, ETA: {eta/60:.1f}min")

        if i<3:
            print(f"    task: {ex['confirmed_task'][:80]}")
            print(f"    pred: {pred_action[:80]}")
            print(f"    gold: {gold_repr[:80]}")

        if (i+1)%10==0:
            gc.collect()
            torch.cuda.empty_cache()

    total_time=time.time()-start_time
    print(f"\nCompleted {total} examples in {total_time/60:.1f} minutes")

    metrics=compute_metrics(predictions)
    metrics["ablation"]=ablation_name
    metrics["mode"]=mode
    metrics["total_time_seconds"]=total_time

    return predictions,metrics

In [20]:

def parse_operation(op_field):
    if isinstance(op_field,str):
        return json.loads(op_field)
    return op_field

def get_op_type(op_dict):
    return op_dict.get("op","CLICK").upper()

def get_op_value(op_dict):
    return op_dict.get("value","")

def truncate_html(html_str,max_chars):
    if len(html_str)<=max_chars:
        return html_str,False
    half=max_chars//2
    return html_str[:half]+"\n... [TRUNCATED] ...\n"+html_str[-half:],True

def get_candidate_actions(example):
    return example["action_reprs"]

def get_target_index(example):
    idx_str=example["target_action_index"]
    try:
        return int(idx_str)
    except:
        return -1

def get_target_repr(example):
    return example["target_action_reprs"]

def get_screenshot(example):
    img=example["screenshot"]
    if isinstance(img,Image.Image):
        return img
    if isinstance(img,dict) and "bytes" in img:
        return Image.open(BytesIO(img["bytes"]))
    if isinstance(img,dict) and "path" in img:
        return Image.open(img["path"])
    return None

def get_task_id(example):
    return example["annotation_id"]


SYSTEM_PROMPT=(
    "You are a web navigation agent. You are given a screenshot of a webpage, "
    "the cleaned HTML of the page, and a list of candidate actions. "
    "Your job is to select the single best action that accomplishes the user's task. "
    "Do NOT invent new actions. You MUST choose from the provided candidates only.\n\n"
    "Respond with exactly:\nAnswer: <index>\n"
    "where <index> is the zero-based integer index of your chosen action."
)

def build_candidate_str(candidates):
    return "\n".join([f"[{i}] {c}" for i,c in enumerate(candidates)])


def build_zeroshot_prompt(example,max_html_chars):
    html_raw=example["cleaned_html"]
    html_text,was_truncated=truncate_html(html_raw,max_html_chars)

    candidates=get_candidate_actions(example)
    cand_str=build_candidate_str(candidates)
    task=example["confirmed_task"]

    trunc_note=""
    if was_truncated:
        trunc_note=f" (truncated from {len(html_raw)} to {max_html_chars} chars)"

    text=(
        f"Task: {task}\n\n"
        f"Cleaned HTML{trunc_note}:\n{html_text}\n\n"
        f"Candidate Actions:\n{cand_str}\n\n"
        f"Select the correct action index. Respond with exactly:\nAnswer: <index>"
    )

    return text,html_text


def build_fewshot_prefix(fewshot_examples_dict,max_html_chars):
    parts=[]
    for op_type in ["CLICK","TYPE","SELECT"]:
        if op_type not in fewshot_examples_dict:
            continue

        info=fewshot_examples_dict[op_type]
        ex=info["example"]

        html_text,_=truncate_html(ex["cleaned_html"],max_html_chars)
        candidates=get_candidate_actions(ex)
        cand_str=build_candidate_str(candidates)
        target_idx=get_target_index(ex)
        task=ex["confirmed_task"]

        parts.append(
            f"--- Example ({op_type}) ---\n"
            f"Task: {task}\n\n"
            f"Cleaned HTML:\n{html_text}\n\n"
            f"Candidate Actions:\n{cand_str}\n\n"
            f"Answer: {target_idx}"
        )

    return "\n\n".join(parts)


def build_fewshot_prompt(example,fewshot_prefix,max_html_chars):
    html_raw=example["cleaned_html"]
    html_text,was_truncated=truncate_html(html_raw,max_html_chars)

    candidates=get_candidate_actions(example)
    cand_str=build_candidate_str(candidates)
    task=example["confirmed_task"]

    trunc_note=""
    if was_truncated:
        trunc_note=f" (truncated from {len(html_raw)} to {max_html_chars} chars)"

    text=(
        f"{fewshot_prefix}\n\n"
        f"--- Now predict ---\n"
        f"Task: {task}\n\n"
        f"Cleaned HTML{trunc_note}:\n{html_text}\n\n"
        f"Candidate Actions:\n{cand_str}\n\n"
        f"Select the correct action index. Respond with exactly:\nAnswer: <index>"
    )

    return text,html_text

## Cell 13: SANITY CHECK (10 EXAMPLES, ZERO-SHOT)


In [21]:
print(next(model.parameters()).device)


cuda:0


In [22]:
sanity_preds, sanity_metrics = run_ablation(
    ds_test, "sanity_check_zero_shot", mode="zero_shot",
    n_examples=CONFIG["sanity_check_n"]
)


for k, v in sanity_metrics.items():
    print(f"  {k}: {v}")

sanity_path = os.path.join(CONFIG["output_dir"], "sanity_check_predictions.json")
with open(sanity_path, "w") as f:
    json.dump(sanity_preds, f, indent=2, default=str)
print(f"\nSanity predictions saved to: {sanity_path}")

print("\n=== Sample outputs for manual inspection ===")
for p in sanity_preds[:3]:
    print(f"\nExample {p['example_index']}:")
    print(f"  Task: {p['confirmed_task'][:100]}")
    print(f"  Gold idx: {p['gold_target_index']}, Pred idx: {p['predicted_index']}")
    print(f"  Gold action: {p['gold_target_action'][:100]}")
    print(f"  Pred action: {p['predicted_action'][:100]}")
    print(f"  Top-3: {p['top3_predicted_indices']}")
    raw_preview = p['raw_model_output'][:300] if isinstance(p['raw_model_output'], str) else str(p['raw_model_output'])[:300]
    print(f"  Raw output preview: {raw_preview}")


Running ablation: sanity_check_zero_shot (zero_shot)
Examples: 10, Scoring: True



  [1/10] ✓ pred=0 gold=0 | 10.0s/ex, ETA: 1.5min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [label]   -> CLICK
    gold: [label]   -> CLICK
  [2/10] ✓ pred=1 gold=1 | 5.4s/ex, ETA: 0.7min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [div]  Andorra -> CLICK
    gold: [div]  Andorra -> CLICK
  [3/10] ✗ pred=3 gold=2 | 3.9s/ex, ETA: 0.5min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [span]  Reggae -> CLICK
    gold: [span]  TikTok Series -> CLICK
  [4/10] ✓ pred=3 gold=3 | 3.1s/ex, ETA: 0.3min
  [5/10] ✗ pred=3 gold=4 | 2.7s/ex, ETA: 0.2min
  [6/10] ✓ pred=5 gold=5 | 2.3s/ex, ETA: 0.2min
  [7/10] ✗ pred=5 gold=6 | 2.1s/ex, ETA: 0.1min
  [8/10] ✓ pred=0 gold=0 | 1.9s/ex, ETA: 0.1min
  [9/10] ✓ pred=1 gold=1 | 1.8s/ex, ETA: 0.0min
  [10/10] ✓ pred=2 gold=2 | 1.7s/ex, ETA: 0.0min

Completed 10 examples in 0.3 minutes
  elem

In [23]:
import os, gc
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## Cell 14: FULL ZERO-SHOT ABLATION


In [24]:
#abLation 1

zs_preds, zs_metrics = run_ablation(
    ds_test, "zero_shot", mode="zero_shot", n_examples=None
)

zs_pred_path = os.path.join(CONFIG["output_dir"], "zero_shot_predictions.json")
with open(zs_pred_path, "w") as f:
    json.dump(zs_preds, f, indent=2, default=str)

zs_metrics_path = os.path.join(CONFIG["output_dir"], "zero_shot_metrics.json")
with open(zs_metrics_path, "w") as f:
    json.dump(zs_metrics, f, indent=2)


for i,j in zs_metrics.items():
    print(f"  {i}:{j}")
print(f"Predictions:{zs_pred_path}")
print(f"Metrics:{zs_metrics_path}")



Running ablation: zero_shot (zero_shot)
Examples: 1019, Scoring: True

  [1/1019] ✓ pred=0 gold=0 | 0.7s/ex, ETA: 12.7min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [label]   -> CLICK
    gold: [label]   -> CLICK
  [2/1019] ✓ pred=1 gold=1 | 0.7s/ex, ETA: 12.6min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [div]  Andorra -> CLICK
    gold: [div]  Andorra -> CLICK
  [3/1019] ✗ pred=3 gold=2 | 0.7s/ex, ETA: 12.7min
    task: What are the romantic reggae musics from BCD Studio that can be used in tik tok 
    pred: [span]  Reggae -> CLICK
    gold: [span]  TikTok Series -> CLICK
  [4/1019] ✓ pred=3 gold=3 | 0.8s/ex, ETA: 12.8min
  [5/1019] ✗ pred=3 gold=4 | 0.8s/ex, ETA: 12.7min
  [6/1019] ✓ pred=5 gold=5 | 0.8s/ex, ETA: 12.8min
  [7/1019] ✗ pred=5 gold=6 | 0.8s/ex, ETA: 12.8min
  [8/1019] ✓ pred=0 gold=0 | 0.8s/ex, ETA: 12.7min
  [9/1019] ✓ pred=1 gold=1 | 0.8s/ex, ETA: 12.

## Cell 15: FULL FEW-SHOT ABLATION


## Cell 16: COMPARISON SUMMARY
